<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/mates/notebooks/c3_l5.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C3-L5 · Correlación y diversificación | Matriz de correlación BTC/ETH/SOL y portafolio 50/50.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red ), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c3_l5.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/mates/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
import numpy as np
corr = df[["btc", "eth", "sol"]].corr()
print(corr.round(2))
c_be = corr.loc["btc", "eth"]
c_bs = corr.loc["btc", "sol"]
port = 0.5 * df["btc"] + 0.5 * df["eth"]
vol = lambda s: s.std(ddof=1) * (252 ** 0.5)
vol_btc, vol_eth, vol_port = vol(df["btc"]), vol(df["eth"]), vol(port)
vol_prom = 0.5 * (vol_btc + vol_eth)
print(f"corr BTC-ETH: {c_be:.2f} | corr BTC-SOL: {c_bs:.2f}")
print(f"vol BTC: {vol_btc:.2%} | vol ETH: {vol_eth:.2%}")
print(f"vol portafolio 50/50: {vol_port:.2%} | promedio: {vol_prom:.2%}")

## Gemelas vs independientes | BTC–ETH se mueven juntas; BTC–SOL va por su cuenta y por eso diversifica.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["BTC-ETH", "BTC-SOL"], [c_be, c_bs], color=["#f87171", "#5eead4"])
ax.axhline(0, color="#52525b", linewidth=1)
ax.set_ylabel("correlación")
ax.set_title("BTC–ETH gemelas (0,87) · BTC–SOL independientes (−0,15)")
plt.show()

In [ ]:
print(f"Alivio por diversificar: {vol_prom - vol_port:.2%} (de {vol_prom:.2%} a {vol_port:.2%})")
print("Conclusión: correlación alta ≈ diversificación de adorno.")

In [ ]:
# Chequeos automáticos
assert len(df) == 50, "se esperan 50 días"
assert c_be > 0.70, "BTC-ETH deben salir gemelas"
assert abs(c_bs) < 0.40, "BTC-SOL debe salir baja"
assert vol_port < vol_prom, "el portafolio debe aliviar algo la volatilidad"
print("Chequeos OK: 0,87 / −0,15 / 49,50% vs 51,17%")